# Aula 1 - Preparação de dados tabulares para classificação

> Este material é exclusivamente didático. Os modelos e resultados não devem ser usados para diagnóstico clínico.

## Dados utilizados

O conjunto de dados utilizado como base foi o [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), que é amplamente utilizado para análise e classificação de câncer de mama.

Ele contém medições computacionais de características extraídas de imagens de massas celulares em exames de mamografia. As características foram calculadas a partir de imagens digitalizadas de aspirações por agulha fina (FNA), sendo usadas para prever se uma massa é maligna (câncer) ou benigna (não cancerígena).

Estes dados foram disponibilizados no repositório [UCI Machine Learning](https://archive.ics.uci.edu/) e originalmente coletados pelo Dr. William H. Wolberg da University of Wisconsin Hospitals, Madison.

## Bibliotecas e reprodutibilidade

In [ ]:
import os
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Aquisição dos dados

Os dados compactados do dataset [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) são baixados diretamente do repositório da [UCI Machine Learning](https://archive.ics.uci.edu/) e descompactados na pasta `/data`.

In [ ]:
# Link para o dataset
DATASET_URL = "https://archive.ics.uci.edu/static/public/17/breast+cancer+wisconsin+diagnostic.zip"

def find_project_root(start=Path.cwd()):
    """Percorre os diretórios pais até encontrar a raiz do projeto (contendo pyproject.toml)."""
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Raiz do projeto não encontrada.")

PROJECT_ROOT = find_project_root()

# Pasta de destino onde o arquivo será descompactado
dest_folder = PROJECT_ROOT / "data"

# Arquivo zip de destino
dest_zip_file = dest_folder / "breast_cancer_wisconsin_diagnostic.zip"

if not dest_folder.exists():

    # Crie a pasta se ela não existir
    print(f"Criando a pasta {dest_folder}...")
    dest_folder.mkdir(parents=True, exist_ok=True)

    print(f"Pasta {dest_folder} criada com sucesso.")

if not dest_zip_file.exists():

    # Faça o download e salve o arquivo
    print("Baixando o dataset...")
    response = requests.get(DATASET_URL, timeout=60)  # Defina um tempo limite de 60 segundos
    response.raise_for_status()  # Levanta um erro se o download falhar

    with open(dest_zip_file, "wb") as f:
        f.write(response.content)
    print(f"Download concluído e salvo em {dest_zip_file}")

    # Descompacte o arquivo
    print("Descompactando o arquivo...")
    with zipfile.ZipFile(dest_zip_file, 'r') as zip_ref:
        zip_ref.extractall(dest_folder)

    print(f"Arquivos descompactados em {dest_folder}")
else:
    print(f"Os arquivos já existem em {dest_folder}")

## Carga e ajuste de colunas

Os arquivos são carregados e um conjunto de 12 colunas são selecionadas e renomeadas para nomes mais descritivos, facilitando a interpretação dos dados.

In [ ]:
# Caminho para os arquivos de dados e nomes
data_file = os.path.join(dest_folder, "wdbc.data")

# Carregar os dados
data = pd.read_csv(data_file, header=None, sep=",")

# 12 primeiras colunas
data = data.iloc[:, :12]

# Definir os nomes das colunas
column_names = [
    "id_number",
    "diagnosis",
    "radius",
    "texture",
    "perimeter",
    "area",
    "smoothness",
    "compactness",
    "concavity",
    "concave points",
    "symmetry",
    "fractal dimension"
]

# Definir os nomes das colunas no DataFrame
data.columns = column_names

# Exibir as primeiras linhas do DataFrame
data.head()

## Distribuição de diagnósticos

Distribuição de diagnósticos entre os casos de câncer de mama maligno e benigno, com base na coluna `diagnosis`.

In [ ]:
# Mostrar informações sobre a coluna "diagnosis"
data["diagnosis"].value_counts()

Distribuição de atributos por diagnóstico, considerando `radius`, `texture`, `perimeter` e `area` segmentados entre os diagnósticos maligno e benigno.

In [ ]:
# Plotar a distribuição dos atributos por diagnóstico
sns.pairplot(data, hue='diagnosis', vars=['radius', 'texture', 'perimeter', 'area'], palette='Set2')
plt.suptitle("Distribuição dos Atributos por Diagnóstico", y=1.02)
plt.show()

## Salvar dados

O DataFrame é salvo em um arquivo parquet na pasta `data`.

In [ ]:
# Arquivo parquet de destino
dest_parquet_file = dest_folder / "breast_cancer.parquet"

# salvar dataframe em um arquivo parquet na pasta data
data.to_parquet(dest_parquet_file)

## Gerar laudos sintéticos baseado nos dados tabulares

Laudos médicos sintéticos são gerados em português, combinando dados fictícios de paciente (via `Faker`) com as características e o diagnóstico de cada exame.

In [ ]:
from faker import Faker
fake = Faker('pt_BR')

# Função para traduzir o diagnóstico
def translate_diagnosis(diagnosis):
    """Traduz o código de diagnóstico ('B'/'M') para o rótulo em português."""
    return "Benigno" if diagnosis == "B" else "Maligno"

# Função para gerar laudo
def generate_report(row):
    """Gera um laudo médico sintético em português a partir dos dados de uma linha do DataFrame."""
    patient = fake.name()
    size = round(row['radius'] * 2, 1)
    margin_texture = "regular" if row['texture'] < 20 else "irregular"
    quadrant = random.choice(["superior esquerdo", "superior direito", "inferior esquerdo", "inferior direito"])
    diagnosis_label = translate_diagnosis(row['diagnosis'])

    return f"""
        Paciente: {patient}
        Exame: Mamografia
        Data do Exame: {fake.date_this_year()}

        Descrição:
        Observou-se uma lesão de aproximadamente {size} mm, localizada no quadrante {quadrant}, com bordas {margin_texture}.
        O exame sugere que a lesão apresenta características {diagnosis_label.lower()}.

        Conclusão: {diagnosis_label}.
        Recomendação: {('Acompanhar evolução com novo exame em 6 meses' if diagnosis_label == 'Benigno' else 'Encaminhar para biópsia e avaliação oncológica')}.
        """


# Aplicar em todas as linhas do dataset
data['report'] = data.apply(generate_report, axis=1)

## Verificar laudos gerados

Conferência dos laudos sintéticos gerados, exibindo exemplos resumidos e o texto completo para casos malignos e benignos.

In [ ]:
# Mostre apenas as colunas diagnosis e report
data[['id_number','diagnosis', 'report']].head()

Exibição do texto completo dos laudos gerados para os 3 primeiros casos de diagnóstico maligno.

In [ ]:
# Mostre o texto completo do laudo para os primeiros 3 diagnósticos malignos
for report in data[data['diagnosis'] == 'M']['report'].head(3):
    print(report)
    print("=" * 80)

Exibição do texto completo dos laudos gerados para os 3 primeiros casos de diagnóstico benigno.

In [ ]:
# Mostre o texto completo do laudo para os primeiros 3 diagnósticos benignos
for report in data[data['diagnosis'] == 'B']['report'].head(3):
    print(report)
    print("=" * 80)

## Salvar dados

O DataFrame com laudos é salvo em um arquivo parquet na pasta `data`.

In [ ]:
# Arquivo parquet de destino
dest_parquet_file = dest_folder / "breast_cancer_report.parquet"

# salvar dataframe em um arquivo parquet na pasta data
data.to_parquet(dest_parquet_file)

## Adicionar Ruido

Adição opcional de ruído aos dados, alterando aleatoriamente o diagnóstico de uma pequena fração das amostras para simular imperfeições reais.

In [ ]:
NOISE = True  # Defina como True para adicionar ruído aos dados

# Introduzir ruído nos dados
def adicionar_ruido(row):
    """Introduz ruído em uma linha do DataFrame, com chance de inverter o diagnóstico."""

    # row['radius'] += random.uniform(-1, 1)  # Pequeno ajuste aleatório no raio
    # row['texture'] += random.uniform(-1, 1)  # Pequeno ajuste aleatório na textura

    if random.random() < 0.05:  # 5% de chance de alterar o diagnóstico
        row['diagnosis'] = "B" if row['diagnosis'] == "M" else "M"

    return row

if NOISE:
    data = data.apply(adicionar_ruido, axis=1)

## Salvar dados

O DataFrame com laudos e ruído é salvo em um arquivo parquet na pasta `data`.

In [ ]:
if NOISE:
    # Arquivo parquet de destino
    dest_parquet_file = dest_folder / "breast_cancer_noise.parquet"

    # salvar dataframe em um arquivo parquet na pasta data
    data.to_parquet(dest_parquet_file)